# ModelSplat — 3DGS Point-Cloud ERP Visualization

**TCC @ UFRGS — ERP 3D Classification**

This notebook visualizes the ShapeSplat/ModelNet10 3D Gaussian Splat dataset
and demonstrates the **cascading-sphere ERP preprocessing concept**.

## Concept

Instead of geometric ray-casting (first/last surface intersection as in HSDC/SWHDC
papers on mesh data), we sample the 3D Gaussian Splat volumetrically:

1. Place a virtual camera at the **opacity-weighted centroid** of the Gaussian cloud.
2. Cast rays outward in all ERP directions (W=512, H=256 pixels).
3. Define **N concentric sphere shells** (radii r₁..rₙ) from the centroid.
4. Assign each Gaussian to its shell by distance from centroid.
5. For each shell, accumulate opacity-weighted density at the corresponding ERP pixel.
6. Output: `[N_shells, H, W]` multi-channel ERP — one density map per shell.

**Note on shell spacing:** This notebook uses a simple linear shell spacing for
visualization.  The production pipeline in `src/preprocessing/radiance_field.py`
uses EgoNeRF exponential spacing (Choi et al., CVPR 2023, §3.2):
`r_s = r_near * (r_far/r_near)^(s/(N-1))`.  Use `gaussian_ply_to_erp()` for
training-ready ERP tensors.

**Dataset:** `gs_data/modelsplat/modelsplat_ply/` — 10 ModelNet10 categories,
binary PLY files with 3DGS properties (position, opacity, scale, rotation, SH color).

In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3D projection
from scipy.ndimage import gaussian_filter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# ── project root on sys.path ─────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── import from src ──────────────────────────────────────────────────────────
from src.preprocessing.ply_loader import load_gaussian_ply
from src.preprocessing.radiance_field import (
    compute_centroid,
    build_ray_directions,
    compute_shell_radii,
)
from src.preprocessing.dataset import MODELNET10_CATEGORIES

# ── Global style ─────────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
matplotlib.rcParams['image.interpolation'] = 'nearest'

# ── Paths & constants ─────────────────────────────────────────────────────────
DATA_ROOT = PROJECT_ROOT / 'gs_data' / 'modelsplat' / 'modelsplat_ply'
H, W = 256, 512
N_SHELLS = 8
MN10_CATEGORIES = list(MODELNET10_CATEGORIES)

np.random.seed(42)
print(f"DATA_ROOT  : {DATA_ROOT.resolve()}")
print(f"ERP size   : H={H}, W={W}")
print(f"N_SHELLS   : {N_SHELLS}")
print(f"Categories : {MN10_CATEGORIES}")

In [ ]:
# ── Helper: spherical coordinate conversion ───────────────────────────────────

def cart_to_spherical(
    xyz: np.ndarray, center: np.ndarray
) -> tuple:
    """Convert Cartesian positions to spherical coords relative to center.

    Returns
    -------
    r     : (N,) radius
    theta : (N,) azimuth in [-pi, pi]
    phi   : (N,) elevation in [-pi/2, pi/2]
    """
    rel = xyz - center                     # (N, 3)
    r = np.linalg.norm(rel, axis=1)        # (N,)
    safe_r = np.where(r < 1e-9, 1e-9, r)
    theta = np.arctan2(rel[:, 1], rel[:, 0])          # azimuth
    phi   = np.arcsin(np.clip(rel[:, 2] / safe_r, -1.0, 1.0))  # elevation
    return r, theta, phi


def spherical_to_erp(
    theta: np.ndarray, phi: np.ndarray, H: int, W: int
) -> tuple:
    """Map spherical angles to ERP pixel coordinates."""
    u = ((theta + np.pi) / (2.0 * np.pi) * W).astype(int) % W
    v = np.clip(
        ((np.pi / 2.0 - phi) / np.pi * H).astype(int),
        0, H - 1,
    )
    return u, v


def cascading_sphere_erp(
    gaussians: dict,
    n_shells: int = 8,
    H: int = 256,
    W: int = 512,
) -> tuple:
    """Generate multi-shell ERP from a 3DGS object (point-cloud approach).

    Uses linear shell spacing for visualization.  Production code uses
    EgoNeRF exponential spacing via compute_shell_radii().

    Returns
    -------
    density_erp  : (n_shells, H, W) float32
    color_erp    : (H, W, 3) float32
    shell_edges  : (n_shells+1,) float32 — linear radial boundaries
    centroid     : (3,) float64
    """
    xyz     = gaussians['xyz']
    opacity = gaussians['opacity']
    rgb     = gaussians['rgb']

    centroid = compute_centroid(xyz, opacity)
    r, theta, phi = cart_to_spherical(xyz, centroid)
    u, v = spherical_to_erp(theta, phi, H, W)

    # Linear shell boundaries (for visualization)
    r_low  = np.percentile(r, 5.0)
    r_high = np.percentile(r, 95.0)
    shell_edges = np.linspace(r_low, r_high, n_shells + 1)

    shell_idx = np.searchsorted(shell_edges[1:-1], r)
    shell_idx = np.clip(shell_idx, 0, n_shells - 1)

    density_erp = np.zeros((n_shells, H, W), dtype=np.float32)
    for s in range(n_shells):
        mask = shell_idx == s
        if mask.any():
            np.add.at(density_erp[s], (v[mask], u[mask]), opacity[mask])

    color_accum  = np.zeros((H, W, 3), dtype=np.float64)
    weight_accum = np.zeros((H, W),    dtype=np.float64)
    for c in range(3):
        np.add.at(color_accum[:, :, c], (v, u), opacity * rgb[:, c])
    np.add.at(weight_accum, (v, u), opacity)

    safe_w = np.where(weight_accum > 0, weight_accum, 1.0)
    color_erp = (color_accum / safe_w[:, :, None]).astype(np.float32)
    color_erp = np.clip(color_erp, 0.0, 1.0)

    return density_erp, color_erp, shell_edges.astype(np.float32), centroid


def find_sample(category: str, split: str = 'train', index: int = 0):
    """Return the PLY path for a given category, split, and index."""
    split_dir = DATA_ROOT / category / split
    if not split_dir.exists():
        return None
    objects = sorted(split_dir.iterdir())
    if index >= len(objects):
        return None
    ply = objects[index] / 'point_cloud.ply'
    return ply if ply.exists() else None


print("Helper functions defined.")

## 1. Load & Inspect a Sample

In [ ]:
sample_ply = find_sample('table', split='train', index=0)
if sample_ply is None:
    candidates = list((DATA_ROOT / 'table' / 'train').rglob('point_cloud.ply'))
    sample_ply = candidates[0] if candidates else None

if sample_ply is None:
    raise FileNotFoundError(f"No table PLY found under {DATA_ROOT}")

print(f"Loading: {sample_ply}")
gs = load_gaussian_ply(sample_ply)

xyz     = gs['xyz']
opacity = gs['opacity']
scale   = gs['scale']
rgb     = gs['rgb']

print()
print(f"{'Property':<30} {'Value'}")
print("-" * 55)
print(f"{'N Gaussians':<30} {gs['n_gaussians']:,}")
print(f"{'X range':<30} [{xyz[:,0].min():.4f}, {xyz[:,0].max():.4f}]")
print(f"{'Y range':<30} [{xyz[:,1].min():.4f}, {xyz[:,1].max():.4f}]")
print(f"{'Z range':<30} [{xyz[:,2].min():.4f}, {xyz[:,2].max():.4f}]")
print(f"{'Opacity mean':<30} {opacity.mean():.4f}")
print(f"{'Opacity min':<30} {opacity.min():.4f}")
print(f"{'Opacity max':<30} {opacity.max():.4f}")
print(f"{'% opacity > 0.5':<30} {100*(opacity>0.5).mean():.1f}%")
scale_mag = scale.mean(axis=1)
print(f"{'Scale mag mean':<30} {scale_mag.mean():.5f}")
print(f"{'Scale mag min':<30} {scale_mag.min():.5f}")
print(f"{'Scale mag max':<30} {scale_mag.max():.5f}")
print(f"{'RGB mean (R)':<30} {rgb[:,0].mean():.4f}")
print(f"{'RGB mean (G)':<30} {rgb[:,1].mean():.4f}")
print(f"{'RGB mean (B)':<30} {rgb[:,2].mean():.4f}")

In [ ]:
# ── Figure: Gaussian Property Distributions ───────────────────────────────────

centroid_sample = compute_centroid(xyz, opacity)
r_from_center, _, _ = cart_to_spherical(xyz, centroid_sample)
scale_mag = scale.mean(axis=1)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

ax = axes[0, 0]
ax.hist(opacity, bins=80, color='steelblue', edgecolor='none', alpha=0.85)
ax.axvline(opacity.mean(), color='crimson', linestyle='--', linewidth=1.5, label=f'mean={opacity.mean():.3f}')
ax.set_xlabel('Opacity (sigmoid)')
ax.set_ylabel('Count')
ax.set_title('Opacity Distribution')
ax.legend()

ax = axes[0, 1]
ax.hist(np.clip(scale_mag, 0, np.percentile(scale_mag, 99)), bins=80,
        color='darkorange', edgecolor='none', alpha=0.85)
ax.axvline(scale_mag.mean(), color='navy', linestyle='--', linewidth=1.5,
           label=f'mean={scale_mag.mean():.5f}')
ax.set_xlabel('Mean scale per Gaussian (exp units)')
ax.set_ylabel('Count')
ax.set_title('Scale Magnitude Distribution (99th pct clipped)')
ax.legend()

ax = axes[1, 0]
ax.hist(r_from_center, bins=80, color='mediumseagreen', edgecolor='none', alpha=0.85)
ax.axvline(r_from_center.mean(), color='darkred', linestyle='--', linewidth=1.5,
           label=f'mean={r_from_center.mean():.3f}')
ax.set_xlabel('Distance from opacity-weighted centroid')
ax.set_ylabel('Count')
ax.set_title('Radius from Centroid')
ax.legend()

ax = axes[1, 1]
colors_ch = ['#e74c3c', '#2ecc71', '#3498db']
labels_ch = ['R', 'G', 'B']
for c_idx, (col, lab) in enumerate(zip(colors_ch, labels_ch)):
    ax.hist(rgb[:, c_idx], bins=80, color=col, alpha=0.55, edgecolor='none', label=lab)
ax.set_xlabel('Channel value (SH DC -> [0,1])')
ax.set_ylabel('Count')
ax.set_title('RGB Channel Distributions')
ax.legend()

fig.suptitle('Gaussian Property Distributions — table sample', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. 3D Gaussian Cloud

In [ ]:
# ── Figure: 3D Gaussian Cloud (two views) ────────────────────────────────────

N_SCATTER = min(5000, gs['n_gaussians'])
idx_sub = np.random.choice(gs['n_gaussians'], N_SCATTER, replace=False)

xyz_sub     = xyz[idx_sub]
rgb_sub     = rgb[idx_sub]
opacity_sub = opacity[idx_sub]

pt_size = (opacity_sub / opacity_sub.max() * 20).clip(1, 20)

fig = plt.figure(figsize=(14, 6))

for ax_idx, (elev, azim, title) in enumerate([
    (20, 45,  'Front view (elev=20, azim=45)'),
    (90,  0,  'Top view (elev=90, azim=0)'),
]):
    ax = fig.add_subplot(1, 2, ax_idx + 1, projection='3d')
    ax.scatter(
        xyz_sub[:, 0], xyz_sub[:, 1], xyz_sub[:, 2],
        c=rgb_sub, s=pt_size, alpha=0.6, linewidths=0,
    )
    ax.scatter(
        [centroid_sample[0]], [centroid_sample[1]], [centroid_sample[2]],
        c='red', s=200, marker='*', zorder=10, label='Centroid',
    )
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.view_init(elev=elev, azim=azim)
    ax.legend(loc='upper right', fontsize=9)

fig.suptitle(
    f'3D Gaussian Splat — table (subsampled to {N_SCATTER:,})',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 2.5 3D Surface Reconstruction (High-opacity Gaussians)

Filtering to the top 30% most opaque Gaussians gives a surface-like representation
of the object. Since 3DGS places high-opacity Gaussians on visible surfaces,
this approximates the object's exterior geometry.

In [ ]:
# ── Figure: 3D Surface-like Representation (High-opacity Gaussians) ──────────
thresh_surf = np.percentile(opacity, 70)
mask_surf   = opacity > thresh_surf
xyz_surf    = xyz[mask_surf]
rgb_surf    = rgb[mask_surf]
opa_surf    = opacity[mask_surf]
n_surf      = mask_surf.sum()

print(f"Surface Gaussians (opacity > {thresh_surf:.3f}): {n_surf:,} / {len(opacity):,}  ({100*mask_surf.mean():.1f}%)")

N_SURF  = min(8000, n_surf)
idx_s   = np.random.choice(n_surf, N_SURF, replace=False)
xyz_s   = xyz_surf[idx_s]
rgb_s   = rgb_surf[idx_s]
opa_s   = opa_surf[idx_s]
pt_s    = (opa_s / opa_s.max() * 15).clip(1, 15)

fig = plt.figure(figsize=(18, 6))
views_surf = [
    (20,  45, 'Front-right'),
    (20, 180, 'Back view'),
    (90,   0, 'Top view'),
]

for ax_i, (elev, azim, title) in enumerate(views_surf):
    ax = fig.add_subplot(1, 3, ax_i + 1, projection='3d')
    ax.scatter(
        xyz_s[:, 0], xyz_s[:, 1], xyz_s[:, 2],
        c=rgb_s, s=pt_s, alpha=0.85, linewidths=0,
    )
    ax.scatter(
        [centroid_sample[0]], [centroid_sample[1]], [centroid_sample[2]],
        c='red', s=200, marker='*', zorder=10, label='Centroid',
    )
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title)
    ax.view_init(elev=elev, azim=azim)
    try:
        ax.set_box_aspect([1, 1, 1])
    except AttributeError:
        pass
    ax.legend(fontsize=9)

fig.suptitle(
    f'3D Surface Representation — table\n'
    f'(High-opacity Gaussians: top 30%, {N_SURF:,} points shown)',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 3. Camera at Object Center & ERP Rays

In [ ]:
# ── Figure: Camera + Ray Grid ─────────────────────────────────────────────────
# Use build_ray_directions from src/preprocessing/radiance_field.py

ray_dirs_flat = build_ray_directions(H, W)           # (H*W, 3)
ray_dirs_grid = ray_dirs_flat.reshape(H, W, 3)       # (H, W, 3)

# Elevation map from ray directions (phi = arcsin(dz))
phi_map = np.arcsin(np.clip(ray_dirs_grid[:, :, 2], -1.0, 1.0))

# Subsample 50 rays for 3D visualization
N_RAYS = 50
flat_idx = np.random.choice(H * W, N_RAYS, replace=False)
rows_r = flat_idx // W
cols_r = flat_idx % W
ray_dx = ray_dirs_grid[rows_r, cols_r, 0]
ray_dy = ray_dirs_grid[rows_r, cols_r, 1]
ray_dz = ray_dirs_grid[rows_r, cols_r, 2]
elev_colors = phi_map[rows_r, cols_r]

elev_norm = (elev_colors - elev_colors.min()) / (elev_colors.max() - elev_colors.min() + 1e-8)
ray_colors = plt.cm.coolwarm(elev_norm)[:, :3]

ray_len = 0.3 * (xyz.max() - xyz.min()).max()

fig = plt.figure(figsize=(14, 6))

# Left: 3D scatter + centroid + rays
ax3d = fig.add_subplot(1, 2, 1, projection='3d')
ax3d.scatter(
    xyz_sub[:, 0], xyz_sub[:, 1], xyz_sub[:, 2],
    c='#aaaaaa', s=3, alpha=0.3, linewidths=0,
)
ax3d.scatter(
    [centroid_sample[0]], [centroid_sample[1]], [centroid_sample[2]],
    c='red', s=300, marker='o', zorder=10, label='Camera (centroid)',
)
for i in range(N_RAYS):
    x0, y0, z0 = centroid_sample
    ax3d.quiver(
        x0, y0, z0,
        ray_dx[i] * ray_len, ray_dy[i] * ray_len, ray_dz[i] * ray_len,
        color=ray_colors[i], arrow_length_ratio=0.1, linewidth=0.8, alpha=0.8,
    )
ax3d.set_xlabel('X')
ax3d.set_ylabel('Y')
ax3d.set_zlabel('Z')
ax3d.set_title('Camera at Centroid — Outward Rays')
ax3d.legend(loc='upper right', fontsize=9)
ax3d.view_init(elev=25, azim=50)

# Right: ERP elevation map
ax2d = fig.add_subplot(1, 2, 2)
im = ax2d.imshow(
    phi_map, cmap='coolwarm', aspect='auto',
    extent=[-180, 180, -90, 90],
    vmin=-np.pi/2, vmax=np.pi/2,
)
cb = plt.colorbar(im, ax=ax2d, fraction=0.03, pad=0.04)
cb.set_label('Elevation phi (rad)')
ax2d.set_xlabel('Azimuth theta (degrees)')
ax2d.set_ylabel('Elevation phi (degrees)')
ax2d.set_title(f'ERP Ray Grid (H={H}, W={W})')
ax2d.set_xticks([-180, -90, 0, 90, 180])
ax2d.set_yticks([-90, -45, 0, 45, 90])

fig.suptitle('Virtual Camera at Object Centroid', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Concentric Sphere Shells (Cascas / Layers)

In [ ]:
# ── Compute shell assignment ──────────────────────────────────────────────────

r_all, theta_all, phi_all = cart_to_spherical(xyz, centroid_sample)

# Linear shell edges (visualization only)
r_low  = np.percentile(r_all, 5.0)
r_high = np.percentile(r_all, 95.0)
shell_edges_sample = np.linspace(r_low, r_high, N_SHELLS + 1)
shell_idx_all = np.searchsorted(shell_edges_sample[1:-1], r_all)
shell_idx_all = np.clip(shell_idx_all, 0, N_SHELLS - 1)

# Compare with EgoNeRF exponential spacing (production)
from src.preprocessing.radiance_field import precompute_gaussian_params, compute_shell_radii as _csr
gs_precomp_sample = precompute_gaussian_params(gs, centroid_sample)
shell_radii_exp = _csr(gs_precomp_sample['r_dist'], n_shells=N_SHELLS)
print(f"Linear shell centers  : {np.round(0.5*(shell_edges_sample[:-1]+shell_edges_sample[1:]), 4)}")
print(f"Exponential shell radii (production): {np.round(shell_radii_exp, 4)}")

cmap_shells = plt.cm.get_cmap('tab10', N_SHELLS)

N_SCATTER_SHELL = min(5000, gs['n_gaussians'])
idx_sh = np.random.choice(gs['n_gaussians'], N_SCATTER_SHELL, replace=False)
xyz_sh  = xyz[idx_sh]
sidx_sh = shell_idx_all[idx_sh]

fig = plt.figure(figsize=(14, 6))

ax3d = fig.add_subplot(1, 2, 1, projection='3d')
for s in range(N_SHELLS):
    mask_s = sidx_sh == s
    if mask_s.any():
        ax3d.scatter(
            xyz_sh[mask_s, 0], xyz_sh[mask_s, 1], xyz_sh[mask_s, 2],
            c=[cmap_shells(s)], s=4, alpha=0.6, linewidths=0,
            label=f'Shell {s+1}',
        )
ax3d.scatter(
    [centroid_sample[0]], [centroid_sample[1]], [centroid_sample[2]],
    c='red', s=200, marker='*', zorder=10, label='Centroid',
)
ax3d.set_xlabel('X')
ax3d.set_ylabel('Y')
ax3d.set_zlabel('Z')
ax3d.set_title('Gaussian Shell Assignment')
ax3d.view_init(elev=20, azim=60)
ax3d.legend(loc='upper right', fontsize=7, ncol=2, markerscale=2)

ax2d = fig.add_subplot(1, 2, 2)
r_max = r_all.max()
thickness = 0.1 * r_max
xz_mask = np.abs(xyz[:, 1] - centroid_sample[1]) < thickness

xyz_xz  = xyz[xz_mask]
sidx_xz = shell_idx_all[xz_mask]

for s in range(N_SHELLS):
    m = sidx_xz == s
    if m.any():
        ax2d.scatter(
            xyz_xz[m, 0], xyz_xz[m, 2],
            c=[cmap_shells(s)], s=8, alpha=0.6, linewidths=0,
            label=f'Shell {s+1}',
        )

angle_arr = np.linspace(0, 2 * np.pi, 200)
for edge in shell_edges_sample:
    ax2d.plot(
        centroid_sample[0] + edge * np.cos(angle_arr),
        centroid_sample[2] + edge * np.sin(angle_arr),
        'k--', linewidth=0.8, alpha=0.5,
    )
ax2d.scatter([centroid_sample[0]], [centroid_sample[2]],
             c='red', s=150, marker='*', zorder=10, label='Centroid')
ax2d.set_xlabel('X')
ax2d.set_ylabel('Z')
ax2d.set_title(f'Cross-section — XZ plane (|Y - Y_c| < {thickness:.3f})')
ax2d.set_aspect('equal')
ax2d.legend(loc='upper right', fontsize=7, ncol=2, markerscale=2)

fig.suptitle(
    f'Concentric Shell Assignment — {N_SHELLS} shells, table',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 5. Cascading ERP Density Maps (One Channel Per Shell)

In [ ]:
# ── Compute full cascading ERP (point-cloud approach) ─────────────────────────
density_erp, color_erp, shell_edges_full, centroid_full = cascading_sphere_erp(
    gs, n_shells=N_SHELLS, H=H, W=W,
)

print(f"density_erp shape : {density_erp.shape}  (shells, H, W)")
print(f"color_erp shape   : {color_erp.shape}  (H, W, 3)")
print(f"Shell edges (r)   : {np.round(shell_edges_full, 4)}")
print(f"Centroid          : {centroid_full.round(4)}")

In [ ]:
# ── Figure: N-Shell ERP Density Maps ─────────────────────────────────────────

n_cols = 4
n_rows = (N_SHELLS + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for s in range(N_SHELLS):
    ax = axes[s]
    d  = gaussian_filter(density_erp[s], sigma=2)
    im = ax.imshow(d, cmap='hot', aspect='auto', interpolation='bilinear')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    r0 = shell_edges_full[s]
    r1 = shell_edges_full[s + 1]
    ax.set_title(f'Shell {s+1}: r in [{r0:.3f}, {r1:.3f}]', fontsize=10)
    ax.set_xlabel('Azimuth (pixels)')
    ax.set_ylabel('Elevation (pixels)')

for s in range(N_SHELLS, len(axes)):
    axes[s].set_visible(False)

fig.suptitle(
    'ERP Density Maps — Cascading Sphere Shells (table)',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 6. ERP Color Projection

In [ ]:
# ── Figure: RGB ERP + False-color composite ───────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(16, 7))

ax = axes[0]
ax.imshow(color_erp, aspect='auto', interpolation='bilinear')
ax.set_title('ERP Color Projection (opacity-weighted RGB)', fontsize=12)
ax.set_xlabel('Azimuth (pixels)')
ax.set_ylabel('Elevation (pixels)')

def _norm_channel(arr: np.ndarray) -> np.ndarray:
    vmax = arr.max()
    return arr if vmax < 1e-8 else arr / vmax

composite = np.zeros((H, W, 3), dtype=np.float32)
for c_idx, shell_i in enumerate([0, 1, 2]):
    d = gaussian_filter(density_erp[shell_i], sigma=2)
    composite[:, :, c_idx] = _norm_channel(d)

d3 = gaussian_filter(density_erp[3], sigma=2)
alpha_ch = _norm_channel(d3)
composite = composite * 0.7 + composite * alpha_ch[:, :, None] * 0.3
composite = np.clip(composite, 0, 1)

ax = axes[1]
ax.imshow(composite, aspect='auto', interpolation='bilinear')
ax.set_title('False-color composite (shells 1-4 as R, G, B + brightness)', fontsize=12)
ax.set_xlabel('Azimuth (pixels)')
ax.set_ylabel('Elevation (pixels)')

fig.suptitle(
    'ERP Projections — table (Gaussian Splat)',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 7. Shell Density Profiles

In [ ]:
# ── Figure: Density vs Radius & Shell Profiles ────────────────────────────────

opacity_per_shell = np.array([
    opacity[(shell_idx_all == s)].sum() for s in range(N_SHELLS)
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(r_all, bins=100, color='steelblue', edgecolor='none', alpha=0.8, label='Gaussians')
for edge in shell_edges_full:
    ax.axvline(edge, color='crimson', linestyle='--', linewidth=1.0, alpha=0.8)
ax.axvline(shell_edges_full[0], color='crimson', linestyle='--', linewidth=1.0,
           label='Shell boundaries (5th-95th pct, linear)')
ax.set_xlabel('Distance from centroid')
ax.set_ylabel('Count')
ax.set_title('Radius Distribution with Shell Boundaries')
ax.legend()

ax = axes[1]
bar_colors = [cmap_shells(s) for s in range(N_SHELLS)]
ax.bar(np.arange(1, N_SHELLS + 1), opacity_per_shell,
       color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Shell index')
ax.set_ylabel('Cumulative opacity')
ax.set_title('Total Opacity per Shell')
ax.set_xticks(np.arange(1, N_SHELLS + 1))
for bar_i, val in enumerate(opacity_per_shell):
    ax.text(bar_i + 1, val + opacity_per_shell.max() * 0.01,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Shell Density Profiles — table', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Multi-Category Gallery

In [ ]:
# ── Precompute ERP for all 10 categories ──────────────────────────────────────
# (this may take ~1-2 minutes depending on PLY sizes)

gallery_rgb  = {}  # category -> color_erp (H, W, 3)
gallery_dens = {}  # category -> density_erp (N, H, W)
gallery_n    = {}  # category -> n_gaussians

for cat in MN10_CATEGORIES:
    ply_path = find_sample(cat, split='train', index=0)
    if ply_path is None:
        print(f"  {cat}: NOT FOUND — skipping")
        continue
    try:
        g = load_gaussian_ply(ply_path)
        d_erp, c_erp, _, _ = cascading_sphere_erp(g, n_shells=N_SHELLS, H=H, W=W)
        gallery_rgb[cat]  = c_erp
        gallery_dens[cat] = d_erp
        gallery_n[cat]    = g['n_gaussians']
        print(f"  {cat:<15} OK  ({g['n_gaussians']:,} Gaussians)")
    except Exception as exc:
        print(f"  {cat}: ERROR — {exc}")

print(f"\nLoaded {len(gallery_rgb)} / {len(MN10_CATEGORIES)} categories.")

In [ ]:
# ── Figure: 10-class RGB ERP Gallery ─────────────────────────────────────────

loaded_cats = list(gallery_rgb.keys())
n_cats = len(loaded_cats)

n_cols_gallery = 5
n_rows_gallery = (n_cats + n_cols_gallery - 1) // n_cols_gallery

fig, axes = plt.subplots(
    n_rows_gallery, n_cols_gallery,
    figsize=(20, 4 * n_rows_gallery),
)
axes = axes.flatten()

for i, cat in enumerate(loaded_cats):
    ax = axes[i]
    ax.imshow(gallery_rgb[cat], aspect='auto', interpolation='bilinear')
    ax.set_title(f'{cat}\n({gallery_n[cat]:,} Gaussians)', fontsize=10)
    ax.axis('off')

for i in range(n_cats, len(axes)):
    axes[i].set_visible(False)

fig.suptitle(
    'ERP Color Projection Gallery — ModelNet10',
    fontsize=16, fontweight='bold',
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure: 10-class Middle-Shell Density Gallery ─────────────────────────────

mid_shell = N_SHELLS // 2

fig, axes = plt.subplots(
    n_rows_gallery, n_cols_gallery,
    figsize=(20, 4 * n_rows_gallery),
)
axes = axes.flatten()

for i, cat in enumerate(loaded_cats):
    ax = axes[i]
    d  = gaussian_filter(gallery_dens[cat][mid_shell], sigma=2)
    im = ax.imshow(d, cmap='hot', aspect='auto', interpolation='bilinear')
    ax.set_title(f'{cat}\n(shell {mid_shell+1}/{N_SHELLS})', fontsize=10)
    ax.axis('off')

for i in range(n_cats, len(axes)):
    axes[i].set_visible(False)

fig.suptitle(
    f'Middle-Shell Density ERP Gallery — ModelNet10 (shell {mid_shell+1} of {N_SHELLS})',
    fontsize=16, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 9. Summary

### What was visualized

This notebook demonstrates the **Cascading Sphere ERP** preprocessing concept
applied to the ShapeSplat/ModelNet10 dataset of 3D Gaussian Splats.

**Preprocessing pipeline demonstrated:**
1. Load Gaussian Splat from PLY via `src.preprocessing.ply_loader.load_gaussian_ply()`
2. Compute opacity-weighted centroid via `src.preprocessing.radiance_field.compute_centroid()`
3. Build ERP ray directions via `src.preprocessing.radiance_field.build_ray_directions()`
4. Divide 3D space into N concentric shells (linear spacing here; exponential in production)
5. Accumulate opacity per shell -> `[N_shells, H, W]` multi-channel ERP tensor

**Production pipeline:** For training-ready ERP tensors with EgoNeRF exponential shell
spacing and full volumetric Gaussian evaluation, use:
```python
from src.preprocessing.radiance_field import gaussian_ply_to_erp
erp = gaussian_ply_to_erp(ply_path, n_shells=8, H=256, W=512)
# returns (8, 256, 512) float32
```

**Key hyperparameter — N_SHELLS:**
The number of shells `N_SHELLS` (here = 8) is the primary design choice.
It is analogous to the number of dilation rates N in the SWHDC paper,
where N=4 was shown to cover 96.85% of the spherical surface area.
For the cascading ERP, `N_SHELLS` controls the depth resolution of the
volumetric representation.  The optimal N is determined by ablation.

**Models trained on this ERP:**
- `HSDCNet(in_channels=8)` — ResNet-34 with HSDC blocks (`src/models/backbones/resnet_hsdc.py`)
- `SWHDCResNet(in_channels=8)` — ResNet-50 with SWHDC blocks (same file)

**Reference targets (original papers, geometric ERP):**
- ResNet-34 + HSDC: 97.1% MN10, 93.9% MN40
- ResNet-50 + SWHDC: 94.1% MN10, 91.9% MN40

Results with 3DGS radiance field ERP constitute the TCC contribution.